# 23 — Bài báo, bước 4: định tuyến theo độ tin cậy và hiệu chỉnh xác suất tầng 1 (máy local)

**Phạm vi**

Phân tích bổ sung theo amendment v4.1, chỉ dùng dự đoán đã lưu: không train lại, không gọi API, không đổi cấu hình hay prompt đã khóa.

**Định tuyến theo độ tin cậy**

Mô hình nhỏ phẳng bốn lớp (PhoBERT; phụ: TF-IDF, ViSoBERT) phân loại trước; review có xác suất lớn nhất dưới θ lấy nhãn của LLM phẳng. Hai điểm vận hành θ_max và θ_98 được chọn trên Validation, seed 42.

**Hiệu chỉnh xác suất tầng 1**

Temperature scaling trên Validation cho từng seed, rồi luật ngưỡng hiện hành (v2.0 + chặn điểm biên v3.5) chọn τ_cal trên xác suất đã hiệu chỉnh của seed 42; tầng 2 giữ nguyên.

**Thứ tự**

Phần A chỉ đọc Validation và lưu lựa chọn kèm SHA-256. Phần B đọc nhãn Hold-out và Lazada để chấm; đây là lần dùng Hold-out thứ ba, được nêu trong threats to validity.

In [1]:
import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import load_frozen
from src.classification.evaluate import FOUR_CLASSES, OTHER_ID, metrics, cascade_labels
from src.classification import paper_analysis as PA
from src.classification import routing_calibration as RC
from src.classification import cost_model as CM
from src.classification.strategy_llm_fewshot.classify import cost_summary
from src.utils.llm import prices
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 200)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
FZ = load_frozen(ROOT)
SEEDS = FZ['tier1']['seeds']

def frozen_yaml(fname, key, must):
    txt = open(os.path.join(ROOT, '03_configs', 'frozen', fname), encoding='utf-8').read()
    assert txt.startswith('# FROZEN'), f'{fname}: thiếu header FROZEN'
    for m in must:
        assert m in txt, f'{fname}: thiếu dấu "{m}"'
    return yaml.safe_load(txt)[key]

TF = frozen_yaml('paper_tfidf_config.yaml', 'tfidf', ['amendment v3.2'])
NC = frozen_yaml('paper_neural_config.yaml', 'paper_neural', ['amendment v3.4', 'amendment v3.5'])
CFG_COST = frozen_yaml('paper_cost_model.yaml', 'cost_model', ['amendment v3.8'])
assert 'v4.1' in open(os.path.join(ROOT, '00_protocol', 'protocol_amendments.md'), encoding='utf-8').read()
FROZEN_THR = {'TF-IDF': TF['tier1']['threshold'], 'Bi-LSTM + fastText': FZ['tier1']['bilstm']['threshold'],
              'ViSoBERT': FZ['tier1']['visobert']['threshold'], 'PhoBERT': NC['phobert']['tier1']['threshold']}
print('Seeds:', SEEDS, '| ngưỡng tầng 1 đã khóa:', FROZEN_THR)

Seeds: [42, 1337, 2024] | ngưỡng tầng 1 đã khóa: {'TF-IDF': 0.45, 'Bi-LSTM + fastText': 0.09, 'ViSoBERT': 0.11, 'PhoBERT': 0.02}


## Phần A — chọn trên Validation

**Dữ liệu**

Chỉ đọc nhãn Validation. Dự đoán Validation của LLM phẳng lấy từ nhật ký gọi API đã lưu; Macro-F1 phải khớp con số 76,1 của notebook 15.

In [2]:
VAL = pd.read_csv(os.path.join(DATA, 'shopee_validation.csv'), encoding='utf-8-sig')
VIDS = VAL['review_id'].tolist()
VY4 = VAL['gold_label'].map(RC.LBL).to_numpy()
VY1 = (VY4 != OTHER_ID).astype(int)
LLM_VAL = RC.flat_llm_validation(os.path.join(RES, 'paper_flat_llm_val_calls.jsonl'), VIDS)
LLM_VAL_F1 = metrics(VY4, LLM_VAL, FOUR_CLASSES)['macro_f1']
assert round(LLM_VAL_F1 * 100, 1) == 76.1, LLM_VAL_F1
print(f'LLM phẳng trên Validation: Macro-F1 {LLM_VAL_F1 * 100:.1f} (khớp notebook 15)')

SMALL_VAL = {x: RC.probabilities(RES, 'validation', VIDS, 'flat', m, SEEDS) for x, m in RC.SMALL.items()}
SEL_ROUTE, SWEEP_VAL = {}, {}
for x, runs in SMALL_VAL.items():
    sw = RC.routing_sweep(VY4, runs[42], LLM_VAL)
    SWEEP_VAL[x] = sw
    SEL_ROUTE[x] = RC.choose_thetas(sw, LLM_VAL_F1)
    base = metrics(VY4, runs[42].argmax(1), FOUR_CLASSES)['macro_f1']
    s = SEL_ROUTE[x]
    print(f"{x}: phẳng {base * 100:.1f} | θ_max = {s['theta_max']:.2f} (Macro-F1 {s['theta_max_val_macro_f1'] * 100:.1f}, "
          f"gửi LLM {s['theta_max_val_llm_rate'] * 100:.1f}%) | θ_98 = {s['theta_98']:.2f} (Macro-F1 {s['theta_98_val_macro_f1'] * 100:.1f}, "
          f"gửi LLM {s['theta_98_val_llm_rate'] * 100:.1f}%, đạt 98%: {s['theta_98_reached']})")

LLM phẳng trên Validation: Macro-F1 76.1 (khớp notebook 15)
PhoBERT: phẳng 66.6 | θ_max = 0.92 (Macro-F1 78.2, gửi LLM 12.3%) | θ_98 = 0.91 (Macro-F1 76.4, gửi LLM 11.1%, đạt 98%: True)


TF-IDF: phẳng 66.5 | θ_max = 0.55 (Macro-F1 78.0, gửi LLM 23.7%) | θ_98 = 0.51 (Macro-F1 75.5, gửi LLM 18.9%, đạt 98%: True)
ViSoBERT: phẳng 68.6 | θ_max = 0.97 (Macro-F1 78.3, gửi LLM 19.4%) | θ_98 = 0.91 (Macro-F1 75.5, gửi LLM 14.0%, đạt 98%: True)


In [3]:
T1_VAL = {x: RC.probabilities(RES, 'validation', VIDS, 'tier1', m, SEEDS) for x, m in RC.TIER1.items()}
SEL_CAL = {}
for x, runs in T1_VAL.items():
    raw = RC.choose_guarded(RC.sweep(VY1, runs[42][:, 1]))['threshold']      # luật trên xác suất gốc của mô hình dùng cho tập khóa
    c = RC.calibrated_threshold(VY1, runs)
    c['raw_rule_threshold'] = raw
    c['ece_val_raw'] = float(np.mean([RC.ece(VY1, p[:, 1]) for p in runs.values()]))
    c['ece_val_cal'] = float(np.mean([RC.ece(VY1, RC.calibrate(p[:, 1], c['temperatures'][s])) for s, p in runs.items()]))
    SEL_CAL[x] = c
    temps = ', '.join('%.2f' % c['temperatures'][s] for s in SEEDS)
    print(f"{x}: ngưỡng đã khóa {FROZEN_THR[x]:.2f} | luật trên xác suất gốc {raw:.2f} ({'khớp' if abs(raw - FROZEN_THR[x]) < 1e-9 else 'KHÁC'}) | T = {temps} | τ_cal {c['tau_cal']:.2f} "
          f"(≈ {c['tau_raw_equivalent_seed42']:.3f} trên xác suất gốc) | recall Validation {c['val_choice']['recall_actionable'] * 100:.1f} | "
          f"ECE Validation {c['ece_val_raw']:.3f} → {c['ece_val_cal']:.3f}")

TF-IDF: ngưỡng đã khóa 0.45 | luật trên xác suất gốc 0.45 (khớp) | T = 0.33, 0.33, 0.33 | τ_cal 0.35 (≈ 0.450 trên xác suất gốc) | recall Validation 80.3 | ECE Validation 0.156 → 0.049


Bi-LSTM + fastText: ngưỡng đã khóa 0.09 | luật trên xác suất gốc 0.09 (khớp) | T = 1.67, 1.98, 1.92 | τ_cal 0.20 (≈ 0.090 trên xác suất gốc) | recall Validation 87.3 | ECE Validation 0.074 → 0.064


ViSoBERT: ngưỡng đã khóa 0.11 | luật trên xác suất gốc 0.11 (khớp) | T = 1.81, 1.82, 1.89 | τ_cal 0.21 (≈ 0.083 trên xác suất gốc) | recall Validation 85.9 | ECE Validation 0.068 → 0.041


PhoBERT: ngưỡng đã khóa 0.02 | luật trên xác suất gốc 0.02 (khớp) | T = 1.70, 1.80, 1.62 | τ_cal 0.09 (≈ 0.019 trên xác suất gốc) | recall Validation 80.3 | ECE Validation 0.078 → 0.073


In [4]:
SELECTION = {'amendment': 'v4.1', 'routing': SEL_ROUTE,
             'calibration': {x: {'temperatures': {str(s): t for s, t in c['temperatures'].items()}, 'tau_cal': c['tau_cal'],
                                 'tau_raw_equivalent_seed42': c['tau_raw_equivalent_seed42']} for x, c in SEL_CAL.items()}}
SEL_PATH = os.path.join(RES, 'paper_step4_selection.json')
SEL_HASH = RC.save_selection(SELECTION, SEL_PATH)
print('Đã lưu lựa chọn trên Validation:', os.path.basename(SEL_PATH), '| SHA-256:', SEL_HASH)

Đã lưu lựa chọn trên Validation: paper_step4_selection.json | SHA-256: c361f95d384e3e1184af560f49723046451ffe513191fa7fa9581bdf0a35ad22


## Phần B — chấm trên các tập khóa

**Ràng buộc**

Lựa chọn của phần A được đọc lại từ file và kiểm tra mã băm trước khi nạp nhãn Hold-out và Lazada; không quyết định nào được điều chỉnh sau bước này.

In [5]:
assert RC.hashlib.sha256(open(SEL_PATH, encoding='utf-8').read().encode('utf-8')).hexdigest() == SEL_HASH
SEL = json.load(open(SEL_PATH, encoding='utf-8'))
NAME = {'holdout': 'shopee_holdout.csv', 'lazada': 'lazada_external_test.csv'}
D = {s: pd.read_csv(os.path.join(DATA, f), encoding='utf-8-sig') for s, f in NAME.items()}
IDS = {s: d['review_id'].tolist() for s, d in D.items()}
Y4 = {s: d['gold_label'].map(RC.LBL).to_numpy() for s, d in D.items()}
PRED = {s: PA.step2_predictions(RES, s, IDS[s], FZ, TF, NC) for s in NAME}
CFG = {s: PA.configurations_step2(*PRED[s])[0] for s in NAME}
REFS = ['Phẳng PhoBERT', 'Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot', 'LLM few-shot → PhoBERT']
ROUTED, RATE = {}, {}
for s in NAME:
    small = {x: RC.probabilities(RES, s, IDS[s], 'flat', m, SEEDS) for x, m in RC.SMALL.items()}
    ROUTED[s], RATE[s] = RC.routed_configs(small, PA.canonical(PRED[s][2][PA.LLM]), SEL['routing'])
print('Cấu hình định tuyến:', list(ROUTED['holdout']))

Cấu hình định tuyến: ['PhoBERT ⇢ LLM (θ_max)', 'PhoBERT ⇢ LLM (θ_98)', 'TF-IDF ⇢ LLM (θ_max)', 'TF-IDF ⇢ LLM (θ_98)', 'ViSoBERT ⇢ LLM (θ_max)', 'ViSoBERT ⇢ LLM (θ_98)']


In [6]:
def routing_tables(s):
    cfg = {**{k: CFG[s][k] for k in REFS}, **ROUTED[s]}
    tab, diff, y, canon = PA.e2e_scores(Y4[s], cfg)
    act = PA.actionable_scores(Y4[s], cfg)
    tab = tab.merge(act, on='cấu hình')
    tab['gửi LLM (%)'] = tab['cấu hình'].map(lambda n: np.mean([m.mean() for m in RATE[s][n].values()]) * 100 if n in RATE[s] else np.nan)
    tab['gửi LLM seed 42 (%)'] = tab['cấu hình'].map(lambda n: RATE[s][n][42].mean() * 100 if n in RATE[s] else np.nan)
    pairs, rows = [], []
    for n in ROUTED[s]:
        x = n.split(' ⇢ ')[0]
        for ref in ['Phẳng ' + x, 'Phẳng LLM few-shot', 'Cascade LLM few-shot', 'LLM few-shot → PhoBERT']:
            pairs.append((n, ref))
    for a, b in pairs:
        d, sg = PA._pair(diff, a, b)
        lo, hi = sorted([sg * d['CI 2,5%'], sg * d['CI 97,5%']])
        n01, n10, p = PA.mcnemar_exact(y, canon[a], canon[b])
        rows.append({'A': a, 'B': b, 'chênh A − B (seed 42)': sg * d['chênh (A − B)'] * 100, 'CI 2,5%': lo * 100, 'CI 97,5%': hi * 100,
                     'A đúng, B sai': n01, 'A sai, B đúng': n10, 'p (exact)': p})
    cmp = pd.DataFrame(rows)
    cmp['p (Holm)'] = PA.holm(cmp['p (exact)'])
    cmp.insert(0, 'họ so sánh', f'định tuyến – {s}')
    return tab, cmp

RT, RCMP = {}, {}
for s in NAME:
    RT[s], RCMP[s] = routing_tables(s)
    v = RT[s][['cấu hình', 'Macro-F1 (4 lớp)', 'CI 2,5%', 'CI 97,5%', 'Recall ACT', 'Precision ACT', 'gửi LLM (%)']].copy()
    v[['CI 2,5%', 'CI 97,5%', 'Recall ACT', 'Precision ACT']] = (v[['CI 2,5%', 'CI 97,5%', 'Recall ACT', 'Precision ACT']] * 100).round(1)
    print(f'Định tuyến — {s} (%):'); print(v.round(1).to_string(index=False)); print()
    print(RCMP[s].round({'chênh A − B (seed 42)': 1, 'CI 2,5%': 1, 'CI 97,5%': 1, 'p (exact)': 4, 'p (Holm)': 4}).to_string(index=False)); print()

Định tuyến — holdout (%):
              cấu hình Macro-F1 (4 lớp)  CI 2,5%  CI 97,5%  Recall ACT  Precision ACT  gửi LLM (%)
         Phẳng PhoBERT       70.9 ± 1.0     64.6      76.0        66.3           77.2          NaN
          Phẳng TF-IDF       62.8 ± 0.0     57.5      67.6        65.6           68.6          NaN
        Phẳng ViSoBERT       66.8 ± 1.5     60.3      72.1        59.6           84.1          NaN
    Phẳng LLM few-shot             80.0     75.1      84.2        90.7           70.3          NaN
  Cascade LLM few-shot             81.3     76.3      85.5        86.9           77.2          NaN
LLM few-shot → PhoBERT             74.6     68.7      79.7        86.9           77.2          NaN
 PhoBERT ⇢ LLM (θ_max)       80.1 ± 0.3     74.8      84.2        75.6           80.6         17.4
  PhoBERT ⇢ LLM (θ_98)       79.7 ± 0.6     73.8      83.6        74.7           81.0         14.8
  TF-IDF ⇢ LLM (θ_max)       76.8 ± 0.0     71.1      81.6        82.0           77

Định tuyến — lazada (%):
              cấu hình Macro-F1 (4 lớp)  CI 2,5%  CI 97,5%  Recall ACT  Precision ACT  gửi LLM (%)
         Phẳng PhoBERT       67.4 ± 2.1     58.4      74.5        74.9           62.7          NaN
          Phẳng TF-IDF       56.1 ± 0.0     48.9      62.6        70.4           50.0          NaN
        Phẳng ViSoBERT       65.3 ± 2.5     57.2      72.9        66.3           65.9          NaN
    Phẳng LLM few-shot             59.5     51.3      66.6        81.5           44.9          NaN
  Cascade LLM few-shot             60.5     52.1      67.7        79.0           50.0          NaN
LLM few-shot → PhoBERT             59.0     50.2      66.6        79.0           50.0          NaN
 PhoBERT ⇢ LLM (θ_max)       64.1 ± 2.6     58.1      74.4        76.1           55.3         28.1
  PhoBERT ⇢ LLM (θ_98)       66.6 ± 2.9     61.1      77.0        77.0           57.2         23.4
  TF-IDF ⇢ LLM (θ_max)       61.7 ± 0.0     53.7      68.7        81.5           51.

In [7]:
CURVE = []
y = Y4['holdout']
llm_h = PA.canonical(PRED['holdout'][2][PA.LLM])
for x, m in RC.SMALL.items():
    p42 = RC.probabilities(RES, 'holdout', IDS['holdout'], 'flat', m, SEEDS)[42]
    sw = RC.routing_sweep(y, p42, llm_h)
    sw.insert(0, 'mô hình nhỏ', x)
    CURVE.append(sw)
CURVE = pd.concat(CURVE, ignore_index=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
REF_F1 = {k: metrics(y, PA.canonical(CFG['holdout'][k]), FOUR_CLASSES)['macro_f1'] * 100 for k in ['Phẳng LLM few-shot', 'Cascade LLM few-shot']}
TXT = {'vi': {'x': 'Tỷ lệ review gửi sang LLM (%)', 'y': 'Macro-F1 end-to-end 4 lớp (%)', 'flat': 'LLM phẳng', 'cas': 'Cascade LLM',
              'tmax': r'$\theta_{\max}$ (chọn trên Validation)', 't98': r'$\theta_{98}$ (chọn trên Validation)'},
       'en': {'x': 'Reviews sent to the LLM (%)', 'y': 'End-to-end four-class macro-F1 (%)', 'flat': 'Flat LLM', 'cas': 'Cascade LLM',
              'tmax': r'$\theta_{\max}$ (chosen on validation)', 't98': r'$\theta_{98}$ (chosen on validation)'}}
for lang, t in TXT.items():
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for i, x in enumerate(RC.SMALL):
        d = CURVE[CURVE['mô hình nhỏ'] == x]
        ax.plot(d['llm_rate'] * 100, d['macro_f1'] * 100, color=CM.SERIES[i], lw=2, label=x)
        for pt, mk in (('theta_max', 'o'), ('theta_98', 's')):
            r = d[np.isclose(d['theta'], SEL['routing'][x][pt])].iloc[0]
            ax.plot(r['llm_rate'] * 100, r['macro_f1'] * 100, mk, color=CM.SERIES[i], ms=7, mec='white', mew=1.2)
    for k, ls, lab in (('Phẳng LLM few-shot', '--', t['flat']), ('Cascade LLM few-shot', ':', t['cas'])):
        ax.axhline(REF_F1[k], color='#52514e', lw=1, ls=ls, label=lab)
    ax.plot([], [], 'o', color='#52514e', label=t['tmax']); ax.plot([], [], 's', color='#52514e', label=t['t98'])
    ax.set_xlabel(t['x'], color='#52514e'); ax.set_ylabel(t['y'], color='#52514e')
    ax.grid(True, color='#e1e0d9', lw=0.6); ax.set_axisbelow(True)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    fig.tight_layout(); fig.savefig(os.path.join(RES, f'paper_step4_routing_curve_{lang}.png'), dpi=200, facecolor='white'); plt.close(fig)
print('Hình: paper_step4_routing_curve_vi.png, paper_step4_routing_curve_en.png')

Hình: paper_step4_routing_curve_vi.png, paper_step4_routing_curve_en.png


In [8]:
def calibrated_cascades(s):
    t1_cal, cas = {}, {}
    for x, m in RC.TIER1.items():
        probs = RC.probabilities(RES, s, IDS[s], 'tier1', m, SEEDS)
        c = SEL['calibration'][x]
        t1_cal[x] = {k: RC.calibrate(p[:, 1], c['temperatures'][str(k)]) >= c['tau_cal'] for k, p in probs.items()}
        cas[f'Cascade {x} (hiệu chỉnh)'] = {k: cascade_labels(a, PRED[s][1][x][k]) for k, a in t1_cal[x].items()}
        cas[f'Cascade {x}'] = CFG[s][f'Cascade {x}']
    return t1_cal, cas

CAL_T1, CAL_E2E, CAL_CMP, ECE_ROWS = {}, {}, {}, []
for s in NAME:
    t1c, cas = calibrated_cascades(s)
    CAL_T1[s] = PA.tier1_scores(Y4[s], {**{x: PRED[s][0][x] for x in RC.TIER1}, **{f'{x} (hiệu chỉnh)': v for x, v in t1c.items()}})[0]
    tab, diff, y, canon = PA.e2e_scores(Y4[s], cas)
    CAL_E2E[s] = tab.merge(PA.actionable_scores(Y4[s], cas), on='cấu hình')
    rows = []
    for x in RC.TIER1:
        a, b = f'Cascade {x} (hiệu chỉnh)', f'Cascade {x}'
        d, sg = PA._pair(diff, a, b)
        lo, hi = sorted([sg * d['CI 2,5%'], sg * d['CI 97,5%']])
        n01, n10, p = PA.mcnemar_exact(y, canon[a], canon[b])
        rows.append({'họ so sánh': f'hiệu chỉnh – {s}', 'chiến lược': x, 'chênh hiệu chỉnh − đã khóa (seed 42)': sg * d['chênh (A − B)'] * 100,
                     'CI 2,5%': lo * 100, 'CI 97,5%': hi * 100, 'A đúng, B sai': n01, 'A sai, B đúng': n10, 'p (exact)': p})
    CAL_CMP[s] = pd.DataFrame(rows)
    CAL_CMP[s]['p (Holm)'] = PA.holm(CAL_CMP[s]['p (exact)'])
    y1 = (Y4[s] != OTHER_ID).astype(int)
    for x, m in RC.TIER1.items():
        probs = RC.probabilities(RES, s, IDS[s], 'tier1', m, SEEDS)
        c = SEL['calibration'][x]
        ECE_ROWS.append({'tập': s, 'chiến lược': x, 'ECE gốc': np.mean([RC.ece(y1, p[:, 1]) for p in probs.values()]),
                         'ECE hiệu chỉnh': np.mean([RC.ece(y1, RC.calibrate(p[:, 1], c['temperatures'][str(k)])) for k, p in probs.items()])})
    print(f'Tầng 1 — {s} (%):'); print(CAL_T1[s][['chiến lược', 'F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)']].to_string(index=False))
    print(f'\nCascade — {s} (%):'); print(CAL_E2E[s][['cấu hình', 'Macro-F1 (4 lớp)', 'Recall ACT', 'Precision ACT']].assign(
        **{'Recall ACT': lambda d: (d['Recall ACT'] * 100).round(1), 'Precision ACT': lambda d: (d['Precision ACT'] * 100).round(1)}).to_string(index=False))
    print(); print(CAL_CMP[s].round(4).to_string(index=False)); print()
ECE = pd.DataFrame(ECE_ROWS)
for x, c in SEL_CAL.items():
    ECE = pd.concat([ECE, pd.DataFrame([{'tập': 'validation', 'chiến lược': x, 'ECE gốc': c['ece_val_raw'], 'ECE hiệu chỉnh': c['ece_val_cal']}])], ignore_index=True)
print('ECE của xác suất ACTIONABLE (trung bình ba seed):'); print(ECE.round(3).to_string(index=False))

Tầng 1 — holdout (%):
                     chiến lược     F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp)
                         TF-IDF 68.1 ± 0.0 78.7 ± 0.0    60.0 ± 0.0       79.8 ± 0.0
             Bi-LSTM + fastText 59.7 ± 4.7 84.0 ± 7.4    47.0 ± 7.5       72.6 ± 5.0
                       ViSoBERT 65.9 ± 0.9 69.8 ± 4.0    62.7 ± 2.9       78.9 ± 0.6
                        PhoBERT 64.0 ± 5.1 81.1 ± 4.1    53.5 ± 8.3       76.4 ± 4.4
            TF-IDF (hiệu chỉnh) 67.9 ± 0.0 78.7 ± 0.0    59.8 ± 0.0       79.6 ± 0.0
Bi-LSTM + fastText (hiệu chỉnh) 59.2 ± 6.3 86.0 ± 6.5    45.8 ± 8.7       71.7 ± 6.6
          ViSoBERT (hiệu chỉnh) 65.2 ± 0.3 71.2 ± 4.6    60.3 ± 2.7       78.2 ± 0.2
           PhoBERT (hiệu chỉnh) 62.9 ± 1.9 81.4 ± 2.2    51.3 ± 3.0       75.7 ± 1.6

Cascade — holdout (%):
                               cấu hình Macro-F1 (4 lớp)  Recall ACT  Precision ACT
            Cascade TF-IDF (hiệu chỉnh)       61.3 ± 0.0        78.7           59.8
                     

Tầng 1 — lazada (%):
                     chiến lược     F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp)
                         TF-IDF 56.9 ± 0.0 88.9 ± 0.0    41.9 ± 0.0       71.2 ± 0.0
             Bi-LSTM + fastText 52.4 ± 6.3 91.8 ± 5.6    37.1 ± 6.7       66.2 ± 7.3
                       ViSoBERT 60.1 ± 1.0 81.1 ± 4.3    47.9 ± 2.7       74.5 ± 1.1
                        PhoBERT 53.0 ± 6.2 88.9 ± 4.3    38.2 ± 6.7       67.3 ± 6.7
            TF-IDF (hiệu chỉnh) 56.9 ± 0.0 88.9 ± 0.0    41.9 ± 0.0       71.2 ± 0.0
Bi-LSTM + fastText (hiệu chỉnh) 50.8 ± 7.9 92.6 ± 4.5    35.5 ± 7.8       64.2 ± 9.6
          ViSoBERT (hiệu chỉnh) 59.7 ± 2.1 84.0 ± 3.7    46.4 ± 3.6       73.8 ± 2.0
           PhoBERT (hiệu chỉnh) 51.5 ± 4.3 89.7 ± 1.4    36.2 ± 4.5       66.0 ± 4.2

Cascade — lazada (%):
                               cấu hình Macro-F1 (4 lớp)  Recall ACT  Precision ACT
            Cascade TF-IDF (hiệu chỉnh)       56.6 ± 0.0        88.9           41.9
                       

In [9]:
COMP = pd.read_csv(os.path.join(RES, 'paper_cost_components.csv'), encoding='utf-8-sig')
MODEL = {'visobert': 'ViSoBERT', 'bilstm': 'Bi-LSTM + fastText', 'tfidf': 'TF-IDF', 'phobert': 'PhoBERT'}
SEC, TRAIN = {}, {}
for f in ('gd10_costs_neural.csv', 'paper_tfidf_costs.csv', 'paper_neural_costs.csv'):
    for _, r in pd.read_csv(os.path.join(RES, f)).iterrows():
        SEC[(r['tier'], MODEL[r['model']])], TRAIN[(r['tier'], MODEL[r['model']])] = r['infer_seconds_per_1000_mean'], r['train_seconds_mean']
SEG = float(pd.read_csv(os.path.join(RES, 'paper_neural_costs.csv')).query("model == 'phobert'")['segment_seconds_per_1000'].iloc[0])
LC = cost_summary(pd.read_csv(os.path.join(RES, 'paper_flat_llm_holdout.csv')), prices(), 'flat LLM')
small_cost = {x: ((SEC[('flat', x)] + (SEG if x == 'PhoBERT' else 0.0)) / 1000, TRAIN[('flat', x)]) for x in RC.SMALL}
names_rates = {n: (n.split(' ⇢ ')[0], float(RATE['holdout'][n][42].mean())) for n in ROUTED['holdout']}
f1_42 = RT['holdout'].set_index('cấu hình')['Macro-F1 chuẩn'].mul(100).to_dict()
RCOMP = RC.routed_cost_rows(names_rates, small_cost, (LC['giây_/1000_review'] / 1000, LC['usd_/1000_review'] / 1000),
                            f1_42, CM.GRID_RUNS, {'PhoBERT': 'GPU', 'ViSoBERT': 'GPU', 'TF-IDF': 'CPU'}, CFG_COST['labels_initial'])
COMP2 = pd.concat([COMP, RCOMP], ignore_index=True)
show = RCOMP.copy(); show['USD API / 1.000 review'] = show.pop('USD API / review') * 1000; show['giây / 1.000 review'] = show.pop('giây suy luận / review') * 1000
print('Thành phần chi phí của các cấu hình định tuyến:'); print(show.round(3).to_string(index=False))

B = CFG_COST['base_scenario']
rows = []
for m in CFG_COST['deployment']:
    for g in CFG_COST['gpu_usd_per_hour']:
        for lc in CFG_COST['label_cost_usd']:
            for n in RCOMP['cấu hình']:
                for b in ('Cascade LLM few-shot', 'Phẳng LLM few-shot'):
                    rows.append({'triển khai': m, 'giá GPU / giờ': g, 'chi phí một nhãn': lc, 'có huấn luyện': n, 'gọi API': b,
                                 'hòa vốn (review / tháng)': CM.break_even(COMP2, CFG_COST, n, b, lc, g, m)})
                rows.append({'triển khai': m, 'giá GPU / giờ': g, 'chi phí một nhãn': lc, 'có huấn luyện': 'Phẳng PhoBERT', 'gọi API': n,
                             'hòa vốn (review / tháng)': CM.break_even(COMP2, CFG_COST, 'Phẳng PhoBERT', n, lc, g, m)})
BE2 = pd.DataFrame(rows)
fmt = lambda v: 'không bao giờ' if np.isinf(v) else f'{v:,.0f}'.replace(',', '.')
t = BE2[(BE2['giá GPU / giờ'] == B['gpu_usd_per_hour']) & (BE2['chi phí một nhãn'] == B['label_cost_usd'])]
print('\nHòa vốn (0,10 USD/nhãn, GPU 0,35 USD/giờ; review / tháng):'); print(t.assign(**{'hòa vốn (review / tháng)': t['hòa vốn (review / tháng)'].map(fmt)}).to_string(index=False))

SC2 = CM.scenarios(COMP2, CFG_COST)
DEC2 = []
for q in CFG_COST['quality_thresholds']:
    for allow in (True, False):
        d = CM.cheapest(SC2, q, allow)
        d.insert(0, 'được gọi API', allow); d.insert(0, 'ngưỡng Macro-F1', q)
        DEC2.append(d)
DEC2 = pd.concat([d for d in DEC2 if len(d)], ignore_index=True)
base = DEC2[(DEC2['giá GPU / giờ'] == B['gpu_usd_per_hour']) & (DEC2['chi phí một nhãn'] == B['label_cost_usd'])]
print('\nCấu hình rẻ nhất đạt ngưỡng khi có thêm cấu hình định tuyến (0,10 USD/nhãn, GPU 0,35 USD/giờ):')
print(base.pivot_table(index=['triển khai', 'review / tháng'], columns=['được gọi API', 'ngưỡng Macro-F1'], values='mã', aggfunc='first').fillna('—').to_string())

Thành phần chi phí của các cấu hình định tuyến:
              cấu hình        mã  kiến trúc  Macro-F1  gọi API  có huấn luyện phần cứng  nhãn ban đầu  lần chạy dò lưới  giây train / lần chạy  USD API / 1.000 review  giây / 1.000 review
 PhoBERT ⇢ LLM (θ_max) P⇢L θ_max định tuyến    79.945     True           True       GPU          1550                 9                 44.462                   0.032               98.579
  PhoBERT ⇢ LLM (θ_98)  P⇢L θ_98 định tuyến    79.126     True           True       GPU          1550                 9                 44.462                   0.028               85.915
  TF-IDF ⇢ LLM (θ_max) T⇢L θ_max định tuyến    76.780     True           True       CPU          1550                18                  0.636                   0.057              170.282
   TF-IDF ⇢ LLM (θ_98)  T⇢L θ_98 định tuyến    73.585     True           True       CPU          1550                18                  0.636                   0.045              133.874
ViSoBERT ⇢ L


Hòa vốn (0,10 USD/nhãn, GPU 0,35 USD/giờ; review / tháng):
triển khai  giá GPU / giờ  chi phí một nhãn          có huấn luyện                gọi API hòa vốn (review / tháng)
     batch           0.35               0.1  PhoBERT ⇢ LLM (θ_max)   Cascade LLM few-shot                   75.149
     batch           0.35               0.1  PhoBERT ⇢ LLM (θ_max)     Phẳng LLM few-shot                   55.118
     batch           0.35               0.1          Phẳng PhoBERT  PhoBERT ⇢ LLM (θ_max)                        0
     batch           0.35               0.1   PhoBERT ⇢ LLM (θ_98)   Cascade LLM few-shot                   72.719
     batch           0.35               0.1   PhoBERT ⇢ LLM (θ_98)     Phẳng LLM few-shot                   53.798
     batch           0.35               0.1          Phẳng PhoBERT   PhoBERT ⇢ LLM (θ_98)                        0
     batch           0.35               0.1   TF-IDF ⇢ LLM (θ_max)   Cascade LLM few-shot                   84.102
     batch          


Cấu hình rẻ nhất đạt ngưỡng khi có thêm cấu hình định tuyến (0,10 USD/nhãn, GPU 0,35 USD/giờ):
được gọi API                 False             True                
ngưỡng Macro-F1                 60       70       60        70   80
triển khai review / tháng                                          
batch      1000            P phẳng  P phẳng      L→L       L→L  L→L
           10000           P phẳng  P phẳng      L→L       L→L  L→L
           100000          P phẳng  P phẳng  P phẳng   P phẳng  L→L
           1000000         P phẳng  P phẳng  P phẳng   P phẳng  L→L
endpoint   1000            T phẳng  P phẳng      L→L       L→L  L→L
           10000           T phẳng  P phẳng      L→L       L→L  L→L
           100000          T phẳng  P phẳng      L→L       L→L  L→L
           1000000         T phẳng  P phẳng  T phẳng  T⇢L θ_98  L→L


In [10]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'paper_step4_{name}.csv'), index=False, encoding='utf-8-sig')

save(pd.concat([sw.assign(**{'mô hình nhỏ': x}) for x, sw in SWEEP_VAL.items()], ignore_index=True), 'routing_sweep_validation')
save(RT['holdout'], 'routing_holdout'); save(RT['lazada'], 'routing_lazada')
save(pd.concat(RCMP.values(), ignore_index=True), 'routing_comparisons'); save(CURVE, 'routing_curve_holdout')
save(CAL_T1['holdout'], 'calibration_tier1_holdout'); save(CAL_T1['lazada'], 'calibration_tier1_lazada')
save(CAL_E2E['holdout'], 'calibration_cascade_holdout'); save(CAL_E2E['lazada'], 'calibration_cascade_lazada')
save(pd.concat(CAL_CMP.values(), ignore_index=True), 'calibration_comparisons'); save(ECE, 'calibration_ece')
save(RCOMP, 'routing_cost_components'); save(BE2, 'routing_break_even'); save(DEC2, 'cost_decision_with_routing')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'python': sys.version.split()[0], 'amendment': 'v4.1', 'selection_sha256': SEL_HASH,
           'n_boot': 10_000, 'bootstrap_seed': 42, 'llm_val_macro_f1': LLM_VAL_F1, 'holdout_use': 3},
          open(os.path.join(RES, 'paper_step4_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('paper_step4_')))

Đã lưu: ['paper_step4_calibration_cascade_holdout.csv', 'paper_step4_calibration_cascade_lazada.csv', 'paper_step4_calibration_comparisons.csv', 'paper_step4_calibration_ece.csv', 'paper_step4_calibration_tier1_holdout.csv', 'paper_step4_calibration_tier1_lazada.csv', 'paper_step4_cost_decision_with_routing.csv', 'paper_step4_routing_break_even.csv', 'paper_step4_routing_comparisons.csv', 'paper_step4_routing_cost_components.csv', 'paper_step4_routing_curve_en.png', 'paper_step4_routing_curve_holdout.csv', 'paper_step4_routing_curve_vi.png', 'paper_step4_routing_holdout.csv', 'paper_step4_routing_lazada.csv', 'paper_step4_routing_sweep_validation.csv', 'paper_step4_run_info.json', 'paper_step4_selection.json']


## Nhận xét

**Định tuyến theo độ tin cậy trên Hold-out**

PhoBERT ⇢ LLM với θ_max = 0,92 đạt Macro-F1 80,1 ± 0,3, ngang LLM phẳng (chênh −0,0; CI −3,8 đến 3,7) và cascade LLM (chênh −1,3; CI −6,0 đến 3,2), cao hơn PhoBERT phẳng 9,1 điểm (CI 4,9 đến 14,0; McNemar Holm p = 0,007). Tỷ lệ review gửi sang LLM là 12,2% ở seed 42 và 17,4% trung bình qua ba seed; chi phí API của lần chạy chuẩn là 0,032 USD cho 1.000 review, khoảng 16% của cascade LLM.

**So với hybrid định tuyến theo lớp**

PhoBERT ⇢ LLM cao hơn LLM → PhoBERT 5,3 điểm (CI −0,0 đến 10,8) với chi phí API thấp hơn nhiều, nên định tuyến theo độ tin cậy giữ được độ chính xác của LLM tốt hơn cách dùng LLM làm tầng lọc. ViSoBERT ⇢ LLM (78,8 ± 1,0) cho kết quả tương tự; TF-IDF ⇢ LLM thấp hơn (76,8) và cần gửi nhiều review hơn (21,5%).

**Trên Lazada**

Với θ chọn trên Validation Shopee, tỷ lệ gửi LLM tăng lên 23–28% vì mô hình nhỏ kém chắc chắn hơn trên ứng dụng mới. PhoBERT ⇢ LLM đạt 64,1 ± 2,6 (θ_max) và 66,6 ± 2,9 (θ_98), không khác PhoBERT phẳng có ý nghĩa thống kê, trong khi θ_98 cao hơn LLM phẳng 10,4 điểm và cascade LLM 9,3 điểm (có ý nghĩa sau hiệu chỉnh Holm). Định tuyến vì vậy giữ được khả năng chuyển ứng dụng của mô hình tinh chỉnh.

**Chi phí**

Khi chạy theo lô với 0,10 USD một nhãn, PhoBERT ⇢ LLM rẻ hơn cascade LLM từ khoảng 73.000–75.000 review mỗi tháng và rẻ hơn LLM phẳng từ khoảng 54.000–55.000; với máy chủ bật liên tục, mức này là khoảng 1,5 triệu. Trong bảng chọn cấu hình theo ngưỡng cứng, PhoBERT ⇢ LLM (79,9 ở seed 42) không qua ngưỡng 80 nên cascade LLM vẫn được chọn cho ngưỡng này; điểm này cần được đọc cùng khoảng tin cậy, vì hai cấu hình không khác nhau có ý nghĩa thống kê.

**Hiệu chỉnh xác suất tầng 1**

Nhiệt độ ước lượng là 0,33 với TF-IDF (xác suất bị kéo về giữa do trọng số lớp cân bằng) và 1,6–2,0 với ba mô hình nơ-ron (quá tự tin). ECE giảm, chẳng hạn TF-IDF từ 0,182 xuống 0,055 và ViSoBERT từ 0,080 xuống 0,043 trên Hold-out, nhưng ngưỡng tương đương trên xác suất gốc gần như không đổi (0,45; 0,09; 0,083 thay cho 0,11; 0,019 thay cho 0,02), nên Macro-F1 cascade chỉ đổi từ −0,8 đến 0,0 điểm, không có ý nghĩa thống kê. Với PhoBERT, τ_cal = 0,09 nằm trong khoảng quét, nên sau hiệu chỉnh luật ngưỡng không còn cần điều kiện chặn điểm biên; điểm vận hành được chọn vẫn gần như cũ.